---
title: "Machine Learning: Transfer, Domain Adaptation, and Multi-Task Learning"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
    code-fold: true
jupyter: python
---


<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/16-transfer-domain-multitask-learning.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Transfer, Domain Adaptation, and Multi-Task Learning**

Most machine-learning systems do not begin with a perfectly matched, independently labeled dataset. A representation may have been pretrained on another task, historical data may come from another population, several related outputs may be available together, or the environment may change after deployment. **Knowledge reuse** asks which information from previous data, tasks, domains, or time periods should influence a new target problem.

A useful formal vocabulary separates a **domain**

$$
\mathcal D=\{\mathcal X,P(X)\}
$$

from a **task**

$$
\mathcal T=\{\mathcal Y,P(Y\mid X)\}.
$$

The source pair $(\mathcal D_S,\mathcal T_S)$ provides data or learned structure; the target pair $(\mathcal D_T,\mathcal T_T)$ defines the deployment objective. Different fields emphasize different changes:

| Setting | Domain | Task | Information flow |
|---|---|---|---|
| Ordinary supervised learning | Same | Same | Train and test under one assumed distribution |
| Transfer learning | May change | May change | Reuse source parameters, features, examples, or priors |
| Domain adaptation | Changes | Usually same or closely related | Adapt a source predictor to a target distribution |
| Multi-task learning | Usually shared or related | Multiple tasks jointly | Learn shared structure at the same time |
| Continual learning | Changes over time | One or many sequential tasks | Acquire new knowledge while retaining old ability |
| Meta-learning | Distribution of domains/tasks | New sampled task | Learn an adaptation rule or initialization |

<div class="diagram-scroll">

![A map of knowledge reuse across domains, tasks, and time.](assets/transfer-learning-map.svg){fig-alt="Source domain and task knowledge flows toward a target, while transfer learning, domain adaptation, multi-task learning, continual learning, and meta-learning occupy different change axes."}

</div>

The words alone do not specify a valid method. Before sharing anything, state:

1. which factor changes: $P(X)$, $P(Y)$, $P(Y\mid X)$, the feature space, or the output space;
2. what source and target labels are available during training;
3. whether target inputs may be inspected before deployment;
4. what is allowed to be stored, replayed, or transferred;
5. what target-only baseline establishes positive or negative transfer.

### **Dataset Shift**

**Dataset shift** occurs when the source and target joint distributions differ:

$$
P_S(X,Y)\ne P_T(X,Y).
$$

The joint distribution has two equivalent factorizations,

$$
P(X,Y)=P(Y\mid X)P(X)=P(X\mid Y)P(Y),
$$

which motivates different shift assumptions. These assumptions matter because unlabeled target inputs reveal $P_T(X)$ but do not identify arbitrary changes in $P_T(Y\mid X)$.

#### **Covariate, Label, and Concept Shift**

**Covariate shift** assumes

$$
P_S(X)\ne P_T(X),
\qquad
P_S(Y\mid X)=P_T(Y\mid X).
$$

The mix of inputs changes, but the conditional decision relationship remains stable. A hospital may see older patients after a service-area change while the disease mechanism conditional on measured variables remains the same. Instance weighting can then make source risk resemble target risk, provided source support covers the target.

**Label shift**, also called prior-probability shift, assumes

$$
P_S(Y)\ne P_T(Y),
\qquad
P_S(X\mid Y)=P_T(X\mid Y).
$$

Class prevalence changes while class-conditional feature patterns remain stable. An epidemic may increase disease prevalence without changing the symptom distribution within each disease class. Posterior probabilities can be corrected if target priors are identifiable.

**Concept shift** or **conditional shift** means

$$
P_S(Y\mid X)\ne P_T(Y\mid X).
$$

The relationship to be predicted has changed. Examples include a fraudster adapting to a detection rule, a policy changing who receives an intervention, or a product definition being revised. Unlabeled target features alone generally cannot reveal this shift; delayed labels, experiments, audits, or structural assumptions are needed.

<div class="diagram-scroll">

![Covariate, label, and concept shift as changes to different factors.](assets/dataset-shift-factorizations.svg){fig-alt="Three panels show changing input density with a stable boundary, changing class prevalence with stable class conditionals, and changing decision relationships."}

</div>

Real systems can experience several shifts simultaneously. The same observed feature drift can be caused by sampling, sensor replacement, changing prevalence, or a new behavioral mechanism. The taxonomy is therefore a model to test, not a diagnosis obtained from a dashboard.


#### **Detecting Distribution Change**

Shift detection should proceed from cheap integrity checks to task-relevant evidence:

- **Schema and data-quality checks:** missingness, type, range, category vocabulary, units, duplication, timestamps, and pipeline version;
- **univariate comparisons:** quantile changes, standardized mean differences, Kolmogorov-Smirnov tests, chi-square tests, or population stability indices;
- **multivariate two-sample tests:** maximum mean discrepancy, energy distance, classifier two-sample tests, or embedding-space distances;
- **prediction monitoring:** class mix, score distribution, entropy, calibration proxies, abstention, and alert volume;
- **delayed performance:** target labels, subgroup metrics, calibration, and utility when outcomes become available.

With large samples, a tiny irrelevant change can be statistically significant. With many features, uncorrected tests produce false alarms. A shift detector should report effect size, uncertainty, multiplicity control, affected subgroups, and operational context.

<div class="diagram-scroll">

![A layered shift-detection and diagnosis workflow.](assets/shift-detection-pipeline.svg){fig-alt="Reference and current windows pass through integrity checks, univariate tests, a domain classifier, representation tests, diagnosis, and an action decision, with delayed labels used for performance."}

</div>

A **domain classifier** labels source observations $d=0$ and target observations $d=1$, then predicts the domain from $x$. Cross-validated AUC near $0.5$ means the chosen features and classifier cannot distinguish the samples; high AUC exposes multivariate shift. It does not prove model performance has degraded.

<details>
<summary><strong>Python: detect multivariate shift with a held-out domain classifier</strong></summary>

```python
import numpy as np
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(160)
source = rng.multivariate_normal(
    mean=[0.0, 0.0, 0.0, 0.0],
    cov=np.array([
        [1.0, 0.7, 0.0, 0.0],
        [0.7, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.2],
        [0.0, 0.0, 0.2, 1.0],
    ]),
    size=1800,
)
target = rng.multivariate_normal(
    mean=[0.0, 0.0, 0.0, 0.0],
    cov=np.array([
        [1.0, -0.45, 0.0, 0.0],
        [-0.45, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.65],
        [0.0, 0.0, 0.65, 1.0],
    ]),
    size=1800,
)

X = np.vstack([source, target])
domain = np.concatenate([np.zeros(len(source)), np.ones(len(target))])
X_train, X_test, d_train, d_test = train_test_split(
    X, domain, test_size=0.35, stratify=domain, random_state=160
)
detector = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LogisticRegression(max_iter=2000),
).fit(X_train, d_train)
domain_auc = roc_auc_score(d_test, detector.predict_proba(X_test)[:, 1])

univariate_p = [ks_2samp(source[:, j], target[:, j]).pvalue for j in range(X.shape[1])]
print("domain-classifier AUC:", round(domain_auc, 3))
print("univariate KS p-values:", np.round(univariate_p, 4).tolist())
```

</details>

Marginal feature tests can miss dependence changes that a multivariate detector finds. Conversely, a flexible domain classifier can exploit harmless identifiers or pipeline artifacts. Inspect its coefficients or explanations and repeat the test by time, site, and subgroup.

Under label shift, a black-box predictor can estimate target class prevalence. Let

$$
C_{ij}=P_S(\hat Y=i\mid Y=j),
\qquad
\mu_i=P_T(\hat Y=i).
$$

If $P(X\mid Y)$ is stable and $C$ is invertible, then $\mu=Cq_T$, so the target prior estimate is $\hat q_T=C^{-1}\hat\mu$. Posterior probabilities can then be adjusted by the prior ratio:

$$
P_T(y\mid x)\propto P_S(y\mid x)\frac{P_T(y)}{P_S(y)}.
$$

<details>
<summary><strong>Python: estimate and correct a simulated label shift</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(161)
means = np.array([[-1.6, 0.0], [0.3, 1.5], [1.7, -0.3]])
covariances = [
    np.array([[1.0, 0.25], [0.25, 0.9]]),
    np.array([[0.9, -0.2], [-0.2, 1.0]]),
    np.array([[1.0, 0.2], [0.2, 0.8]]),
]

def sample_with_prior(prior, size):
    labels = rng.choice(3, size=size, p=prior)
    features = np.vstack([
        rng.multivariate_normal(means[label], covariances[label])
        for label in labels
    ])
    return features, labels

source_prior = np.array([0.60, 0.30, 0.10])
target_prior = np.array([0.20, 0.30, 0.50])
X_train, y_train = sample_with_prior(source_prior, 5000)
X_validation, y_validation = sample_with_prior(source_prior, 3000)
X_target, y_target = sample_with_prior(target_prior, 4000)

model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2500)
).fit(X_train, y_train)

# confusion_matrix returns true-label rows and predicted-label columns.
source_prediction = model.predict(X_validation)
C = confusion_matrix(
    y_validation, source_prediction, labels=[0, 1, 2], normalize="true"
).T
target_prediction = model.predict(X_target)
mu = np.bincount(target_prediction, minlength=3) / len(target_prediction)
estimated_target_prior = np.linalg.solve(C, mu)
estimated_target_prior = np.clip(estimated_target_prior, 1e-6, None)
estimated_target_prior /= estimated_target_prior.sum()

source_probability = model.predict_proba(X_target)
prior_ratio = estimated_target_prior / source_prior
corrected_probability = source_probability * prior_ratio
corrected_probability /= corrected_probability.sum(axis=1, keepdims=True)

print("true target prior:", np.round(target_prior, 3).tolist())
print("estimated target prior:", np.round(estimated_target_prior, 3).tolist())
print("uncorrected accuracy:", round(accuracy_score(y_target, source_probability.argmax(axis=1)), 3))
print("label-shift-corrected accuracy:", round(accuracy_score(y_target, corrected_probability.argmax(axis=1)), 3))
```

</details>

An unstable or nearly singular confusion matrix makes prevalence estimates noisy. Calibration, regularization, soft predictions, and uncertainty intervals help, but no algebra repairs a violated $P_S(X\mid Y)=P_T(X\mid Y)$ assumption.


### **Transfer Learning**

Transfer learning uses knowledge acquired from one source problem to improve a target problem. The transferred object can be training examples, feature transformations, model parameters, a prior distribution, a similarity metric, an optimizer, or an architecture. Modern pretraining followed by task-specific adaptation is one important case, not the whole field.

#### **Feature Reuse and Fine-Tuning**

Suppose a source model decomposes into an encoder $z=f_{\theta}(x)$ and source head $h_{\phi_S}(z)$. Target adaptation replaces or extends the head with $h_{\phi_T}$:

$$
\hat y_T=h_{\phi_T}(f_{\theta}(x_T)).
$$

Three common strategies provide increasing target-specific freedom:

- **frozen feature reuse / linear probe:** keep $\theta$ fixed and train only $\phi_T$;
- **partial fine-tuning:** unfreeze upper encoder blocks while preserving lower features;
- **full fine-tuning:** update all parameters, usually with a smaller learning rate for pretrained weights.

<div class="diagram-scroll">

![A staged strategy from a frozen probe to full fine-tuning.](assets/transfer-finetuning-stages.svg){fig-alt="A pretrained encoder first receives a new frozen target head, then upper blocks are unfrozen, and finally full fine-tuning is considered if target validation supports greater freedom."}

</div>

Freezing reduces variance and computation when target labels are scarce, but creates bias if source features omit target-relevant information. Full fine-tuning increases adaptability but can overfit, destroy useful representations, or forget source behavior. Practical controls include discriminative learning rates, gradual unfreezing, early stopping, weight decay toward pretrained parameters, adapters, low-rank updates, and replay of source examples when permitted.

<details>
<summary><strong>Python: pretrain, probe, and fine-tune a small target classifier</strong></summary>

```python
import copy
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn

torch.manual_seed(162)
np.random.seed(162)
X_raw, digit = load_digits(return_X_y=True)
source_id, target_pool_id = train_test_split(
    np.arange(len(X_raw)), test_size=0.30, stratify=digit, random_state=162
)
scaler = StandardScaler().fit(X_raw[source_id])
X_source = torch.tensor(scaler.transform(X_raw[source_id]).astype("float32"))
y_source = torch.tensor(digit[source_id], dtype=torch.long)
X_target = scaler.transform(X_raw[target_pool_id]).astype("float32")
y_target = (digit[target_pool_id] % 2).astype("int64")

target_train_id, target_test_id = train_test_split(
    np.arange(len(X_target)), train_size=24, stratify=y_target, random_state=162
)
X_target_train = torch.tensor(X_target[target_train_id])
y_target_train = torch.tensor(y_target[target_train_id], dtype=torch.long)
X_target_test = torch.tensor(X_target[target_test_id])
y_target_test = torch.tensor(y_target[target_test_id], dtype=torch.long)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(64, 48), nn.ReLU(), nn.Linear(48, 24), nn.ReLU())

    def forward(self, x):
        return self.layers(x)

def train_model(model, X_train, y_train, epochs, optimizer):
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()

def accuracy(model, X_eval, y_eval):
    with torch.no_grad():
        return float((model(X_eval).argmax(dim=1) == y_eval).float().mean())

# Source pretraining predicts ten-way digit identity.
encoder = Encoder()
source_model = nn.Sequential(encoder, nn.Linear(24, 10))
train_model(
    source_model, X_source, y_source, epochs=160,
    optimizer=torch.optim.Adam(source_model.parameters(), lr=0.01),
)

# Stage 1: freeze the encoder and train a new even-versus-odd head.
transferred_encoder = copy.deepcopy(encoder)
for parameter in transferred_encoder.parameters():
    parameter.requires_grad = False
target_model = nn.Sequential(transferred_encoder, nn.Linear(24, 2))
train_model(
    target_model, X_target_train, y_target_train, epochs=120,
    optimizer=torch.optim.Adam(target_model[1].parameters(), lr=0.03),
)
probe_accuracy = accuracy(target_model, X_target_test, y_target_test)

# Stage 2: unfreeze with a smaller encoder learning rate.
for parameter in target_model[0].parameters():
    parameter.requires_grad = True
optimizer = torch.optim.Adam([
    {"params": target_model[0].parameters(), "lr": 0.001},
    {"params": target_model[1].parameters(), "lr": 0.008},
])
train_model(target_model, X_target_train, y_target_train, epochs=60, optimizer=optimizer)
fine_tuned_accuracy = accuracy(target_model, X_target_test, y_target_test)

# Target-only baseline with the same architecture and label budget.
scratch_model = nn.Sequential(Encoder(), nn.Linear(24, 2))
train_model(
    scratch_model, X_target_train, y_target_train, epochs=180,
    optimizer=torch.optim.Adam(scratch_model.parameters(), lr=0.01),
)

print("frozen-probe accuracy:", round(probe_accuracy, 3))
print("fine-tuned accuracy:", round(fine_tuned_accuracy, 3))
print("target-only scratch accuracy:", round(accuracy(scratch_model, X_target_test, y_target_test), 3))
```

</details>

The target asks a new parity task on held-out observations, so transfer can help only through digit structure learned from the disjoint source split. Because 24 target examples create high variance, repeat the split and report a distribution rather than treating one seed as conclusive.

#### **Positive and Negative Transfer**

**Positive transfer** improves target learning relative to a target-only procedure with the same target labels, compute, and tuning budget. **Negative transfer** makes it worse. For a score where larger is better,

$$
\Delta_{\text{transfer}}(n_T)
=S_{\text{transfer}}(n_T)-S_{\text{target-only}}(n_T).
$$

Transfer gain should be reported across target-label budgets. A source may help with five labels but constrain performance once the target has enough data.

<div class="diagram-scroll">

![Positive and negative transfer and their required baseline.](assets/negative-transfer-map.svg){fig-alt="Source features, parameters, or examples can produce positive or negative transfer, which is identified against a target-only baseline and mitigated through source selection or weaker sharing."}

</div>

Negative transfer arises when source and target semantics differ, source features encode spurious correlations, the source dominates optimization, the target lies outside source support, or the shared model lacks capacity for both. It can be mitigated by source-domain selection, similarity-aware weighting, adapters, task-specific normalization, gating, partial sharing, regularization toward target evidence, or abandoning transfer.

A simple parameter-transfer estimator makes the trade-off visible:

$$
\hat\beta_T
=\arg\min_\beta
\lVert y_T-X_T\beta\rVert_2^2
+\lambda\lVert\beta-\hat\beta_S\rVert_2^2.
$$

The source parameter acts as a prior. It reduces variance when aligned and creates bias when wrong.

<details>
<summary><strong>Python: demonstrate positive and negative parameter transfer</strong></summary>

```python
import numpy as np
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(163)
dimension = 12
beta_target = rng.normal(size=dimension)
beta_target /= np.linalg.norm(beta_target)
beta_aligned = beta_target + 0.15 * rng.normal(size=dimension)
beta_opposite = -beta_target + 0.15 * rng.normal(size=dimension)

def sample_regression(beta, size):
    X = rng.normal(size=(size, dimension))
    y = X @ beta + rng.normal(scale=0.8, size=size)
    return X, y

X_target, y_target = sample_regression(beta_target, 24)
X_test, y_test = sample_regression(beta_target, 3000)
ridge = 2.0
target_only = np.linalg.solve(
    X_target.T @ X_target + ridge * np.eye(dimension),
    X_target.T @ y_target,
)

def transfer_from(source_beta, strength=12.0):
    return np.linalg.solve(
        X_target.T @ X_target + strength * np.eye(dimension),
        X_target.T @ y_target + strength * source_beta,
    )

models = {
    "target only": target_only,
    "aligned source prior": transfer_from(beta_aligned),
    "opposite source prior": transfer_from(beta_opposite),
}
for name, coefficient in models.items():
    print(name, "test MSE:", round(mean_squared_error(y_test, X_test @ coefficient), 3))
```

</details>

Source size does not appear in this toy calculation because the source parameter is supplied directly. In a real system, its uncertainty, provenance, and similarity to the target should control transfer strength.


### **Domain Adaptation**

Domain adaptation targets the same or a closely related prediction problem when source and target distributions differ. A standard setting provides labeled source data

$$
\{(x_i^S,y_i^S)\}_{i=1}^{n_S}
$$

and either labeled or unlabeled target inputs

$$
\{x_j^T\}_{j=1}^{n_T}.
$$

The method must state what target information is available. Using target labels to choose an allegedly unsupervised adaptation hyperparameter converts the experiment into supervised adaptation.

#### **Instance Reweighting**

Under covariate shift, target risk can be rewritten as a weighted source expectation:

$$
\begin{aligned}
R_T(f)
&=\mathbb E_{(X,Y)\sim P_T}[\ell(f(X),Y)]\\
&=\mathbb E_{(X,Y)\sim P_S}
\left[
\frac{p_T(X)}{p_S(X)}
\ell(f(X),Y)
\right].
\end{aligned}
$$

The importance weight is $w(x)=p_T(x)/p_S(x)$. Direct high-dimensional density estimation is difficult, so a domain classifier is often used. If $d=1$ denotes target and the source and target samples used to fit the classifier have sizes $n_S,n_T$,

$$
w(x)
=\frac{P(d=1\mid x)}{P(d=0\mid x)}
\frac{n_S}{n_T}.
$$

<div class="diagram-scroll">

![Instance weighting under covariate shift.](assets/importance-weighting-domain.svg){fig-alt="Labeled source samples receive target-to-source density-ratio weights, causing target-like source examples to contribute more to a weighted source risk."}

</div>

Very large weights indicate weak source support and produce high-variance estimates. Useful diagnostics include weight histograms, clipping sensitivity, overlap plots, and effective sample size

$$
n_{\text{eff}}
=\frac{(\sum_i w_i)^2}{\sum_i w_i^2}.
$$

<details>
<summary><strong>Python: estimate density ratios with a domain classifier</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(164)

def generate(mean, size):
    X = rng.multivariate_normal(mean, [[1.0, 0.25], [0.25, 1.0]], size=size)
    logit = 0.2 + 0.8 * X[:, 0] - 0.6 * X[:, 1] + 1.4 * X[:, 0] * X[:, 1]
    probability = 1 / (1 + np.exp(-logit))
    y = rng.binomial(1, probability)
    return X, y

X_source, y_source = generate(mean=[-0.8, -0.5], size=5000)
X_target_unlabeled, _ = generate(mean=[0.8, 0.7], size=3500)
X_target_test, y_target_test = generate(mean=[0.8, 0.7], size=4000)

domain_X = np.vstack([X_source, X_target_unlabeled])
domain_y = np.concatenate([
    np.zeros(len(X_source)), np.ones(len(X_target_unlabeled))
])
domain_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000)
).fit(domain_X, domain_y)
target_odds = domain_model.predict_proba(X_source)
weights = (
    target_odds[:, 1] / np.clip(target_odds[:, 0], 1e-6, None)
    * len(X_source) / len(X_target_unlabeled)
)
weights = np.clip(weights, 0.05, 20.0)

# A linear task model is misspecified because the true relationship has an interaction.
unweighted = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000)
).fit(X_source, y_source)
weighted = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000)
).fit(X_source, y_source, logisticregression__sample_weight=weights)

for name, model in [("unweighted", unweighted), ("importance weighted", weighted)]:
    probability = model.predict_proba(X_target_test)[:, 1]
    print(
        name,
        "target log loss:", round(log_loss(y_target_test, probability), 3),
        "AUC:", round(roc_auc_score(y_target_test, probability), 3),
    )

effective_size = weights.sum() ** 2 / np.sum(weights ** 2)
print("effective source sample size:", round(float(effective_size), 1), "of", len(weights))
```

</details>

Importance weighting does not change a perfectly specified conditional model asymptotically. It matters here because the linear predictor is misspecified, so emphasizing target-like source regions changes the best approximation for target risk.

#### **Feature Alignment**

Feature alignment learns or transforms a representation $z=f_\theta(x)$ so that source and target feature distributions are closer while source labels remain predictable. Common approaches include:

- **CORAL:** match feature means and covariance;
- **maximum mean discrepancy (MMD):** minimize a kernel two-sample distance;
- **optimal transport:** move source probability mass toward target geometry;
- **domain-adversarial learning:** train an encoder that supports task prediction but confuses a domain classifier;
- **class-conditional alignment:** align within inferred or labeled classes rather than only marginally.

<div class="diagram-scroll">

![Source and target feature alignment in a shared representation.](assets/domain-feature-alignment.svg){fig-alt="Labeled source and unlabeled target features enter a shared representation where covariance, MMD, transport, or adversarial losses align domains while retaining task discrimination."}

</div>

CORAL transforms centered source features using

$$
\tilde X_S
=(X_S-\mu_S)
(C_S+\epsilon I)^{-1/2}
(C_T+\epsilon I)^{1/2}
+\mu_T.
$$

It matches first and second moments without target labels. Eigenvalue regularization $\epsilon$ prevents unstable inverse square roots.

<details>
<summary><strong>Python: implement CORAL and compare adaptation baselines</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(165)

def latent_sample(size):
    y = rng.integers(0, 2, size=size)
    z = rng.normal(size=(size, 3))
    z[:, 0] += np.where(y == 1, 1.3, -1.3)
    z[:, 1] += np.where(y == 1, 0.5, -0.5)
    return z, y

source_z, y_source = latent_sample(2500)
target_z, y_target = latent_sample(1800)
source_transform = np.eye(3)
target_transform = np.array([[1.8, 0.6, 0.0], [0.6, 0.9, 0.1], [0.0, 0.1, 1.3]])
X_source = source_z @ source_transform + np.array([-0.8, 0.4, 0.0])
X_target = target_z @ target_transform + np.array([1.0, -0.5, 0.4])

def symmetric_power(matrix, power, epsilon=1e-5):
    eigenvalue, eigenvector = np.linalg.eigh(matrix + epsilon * np.eye(matrix.shape[0]))
    return eigenvector @ np.diag(eigenvalue ** power) @ eigenvector.T

def coral_source_to_target(source, target):
    source_centered = source - source.mean(axis=0)
    target_mean = target.mean(axis=0)
    target_centered = target - target_mean
    C_source = np.cov(source_centered, rowvar=False)
    C_target = np.cov(target_centered, rowvar=False)
    return (
        source_centered
        @ symmetric_power(C_source, -0.5)
        @ symmetric_power(C_target, 0.5)
        + target_mean
    )

source_only = LogisticRegression(max_iter=2000).fit(X_source, y_source)
X_source_coral = coral_source_to_target(X_source, X_target)
coral_model = LogisticRegression(max_iter=2000).fit(X_source_coral, y_source)

# A few labeled target examples define a supervised-adaptation reference.
few_target = np.concatenate([
    rng.choice(np.flatnonzero(y_target == class_id), 12, replace=False)
    for class_id in [0, 1]
])
supervised_adaptation = LogisticRegression(max_iter=2000).fit(
    np.vstack([X_source_coral, X_target[few_target]]),
    np.concatenate([y_source, y_target[few_target]]),
    sample_weight=np.concatenate([np.ones(len(y_source)), np.full(len(few_target), 15.0)]),
)

for name, model in [
    ("source only", source_only),
    ("unsupervised CORAL", coral_model),
    ("CORAL plus 24 target labels", supervised_adaptation),
]:
    print(name, "target accuracy:", round(accuracy_score(y_target, model.predict(X_target)), 3))
```

</details>

Moment matching can align the wrong classes. In the domain-adversarial objective,

$$
\min_{\theta,\phi}\max_{\psi}
\quad
\mathcal L_{\text{task}}(h_\phi(f_\theta(x_S)),y_S)
-\lambda\mathcal L_{\text{domain}}(d_\psi(f_\theta(x)),d),
$$

the encoder minimizes source task loss while maximizing domain-classifier loss, usually through a gradient-reversal layer. This encourages domain-invariant features, but invariance is useful only when one classifier can perform well in both domains. A classic adaptation bound reflects this:

$$
\epsilon_T(h)
\le
\epsilon_S(h)
+\frac{1}{2}d_{\mathcal H\Delta\mathcal H}(P_S(X),P_T(X))
+\lambda^\star,
$$

where $\lambda^\star$ is the error of the best shared hypothesis. Reducing marginal divergence does not reduce a large $\lambda^\star$ caused by conditional or semantic mismatch.

#### **Supervised and Unsupervised Adaptation**

Adaptation settings should be named by target information:

| Setting | Target inputs | Target labels | Typical methods |
|---|---|---|---|
| Supervised domain adaptation | Available | A labeled target sample | Fine-tuning, joint training, target-weighted loss |
| Semi-supervised adaptation | Available | Few labels plus many unlabeled | Pseudo-labels, consistency, conditional alignment |
| Unsupervised domain adaptation | Available | None during training or selection | Reweighting, CORAL, MMD, DANN |
| Source-free adaptation | Available | None, and source data unavailable | Adapt a supplied source model using target data |
| Domain generalization | Target unavailable during training | None | Learn across several sources for an unseen domain |
| Test-time adaptation | Streaming test batches | Usually none | Update normalization, entropy, or self-supervised objectives |

Unsupervised does not mean evaluation without target labels. Target labels are still required after development to establish whether adaptation helped. They must remain hidden from training, hyperparameter selection, and stopping.


### **Multi-Task Learning**

Multi-task learning jointly optimizes related tasks so their training signals act as inductive bias. If tasks depend on shared latent factors, one task can regularize another, expand effective data, and improve representation learning. If tasks conflict, sharing creates interference and negative transfer.

For tasks $t=1,\ldots,T$ with losses $\mathcal L_t$, the basic objective is

$$
\min_\theta \sum_{t=1}^{T}\lambda_t\mathcal L_t(\theta,\phi_t),
$$

where $\theta$ denotes shared parameters, $\phi_t$ task-specific parameters, and $\lambda_t$ task weights.

#### **Hard and Soft Parameter Sharing**

**Hard parameter sharing** uses one common encoder with separate heads:

$$
z=f_\theta(x),
\qquad
\hat y_t=h_{\phi_t}(z).
$$

It is parameter-efficient and strongly regularizing, but forces all tasks through the same bottleneck. **Soft parameter sharing** keeps task-specific models and couples them:

$$
\min_{\theta_1,\theta_2}
\mathcal L_1(\theta_1)+\mathcal L_2(\theta_2)
+\lambda\lVert\theta_1-\theta_2\rVert_2^2.
$$

More flexible variants learn a shared low-rank basis, cross-stitch feature combinations, gates, adapters, or mixtures of experts.

<div class="diagram-scroll">

![Hard and soft parameter sharing for multi-task learning.](assets/multitask-sharing.svg){fig-alt="Hard sharing routes task inputs through one common trunk and separate heads, while soft sharing keeps private models coupled by a similarity or low-rank regularizer."}

</div>

<details>
<summary><strong>Python: train a shared trunk with two task-specific heads</strong></summary>

```python
import numpy as np
import torch
from sklearn.metrics import mean_squared_error
from torch import nn

torch.manual_seed(166)
rng = np.random.default_rng(166)
X_train = rng.uniform(-3, 3, size=(140, 1)).astype("float32")
X_test = np.linspace(-3, 3, 1000, dtype="float32").reshape(-1, 1)

def task_targets(X, noise=True):
    scale = 0.18 if noise else 0.0
    y1 = np.sin(X[:, 0]) + rng.normal(0, scale, len(X))
    y2 = np.sin(X[:, 0]) + 0.35 * X[:, 0] + rng.normal(0, scale, len(X))
    return y1.astype("float32"), y2.astype("float32")

y1_train, y2_train = task_targets(X_train)
y1_test = np.sin(X_test[:, 0])
y2_test = np.sin(X_test[:, 0]) + 0.35 * X_test[:, 0]
X_tensor = torch.tensor(X_train)

class MultiTaskNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(1, 24), nn.Tanh(), nn.Linear(24, 16), nn.Tanh())
        self.head1 = nn.Linear(16, 1)
        self.head2 = nn.Linear(16, 1)

    def forward(self, x):
        z = self.trunk(x)
        return self.head1(z).squeeze(1), self.head2(z).squeeze(1)

model = MultiTaskNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.015, weight_decay=1e-4)
targets = [torch.tensor(y1_train), torch.tensor(y2_train)]
for _ in range(700):
    optimizer.zero_grad()
    prediction = model(X_tensor)
    loss = sum(nn.functional.mse_loss(prediction[i], targets[i]) for i in range(2))
    loss.backward()
    optimizer.step()

with torch.no_grad():
    prediction1, prediction2 = model(torch.tensor(X_test))
print("task 1 test MSE:", round(mean_squared_error(y1_test, prediction1.numpy()), 4))
print("task 2 test MSE:", round(mean_squared_error(y2_test, prediction2.numpy()), 4))
print("shared parameters:", sum(p.numel() for p in model.trunk.parameters()))
print("task-specific parameters:", sum(p.numel() for p in model.head1.parameters()) * 2)
```

</details>

The two tasks share a sinusoidal component, so a common trunk is plausible. A fair transfer claim still requires separately trained single-task networks with matched tuning and capacity; this code demonstrates the architecture rather than claiming universal improvement.

#### **Task Weighting and Interference**

Loss weights do more than express importance. They set gradient scale:

$$
g=\sum_t\lambda_t g_t,
\qquad
g_t=\nabla_\theta\mathcal L_t.
$$

One task may dominate because its loss has different units, more examples, larger noise, or faster learning dynamics. Static normalization, uncertainty weighting, dynamic weight averaging, and GradNorm attempt to balance scales or training rates.

Tasks interfere when

$$
g_i^\top g_j<0,
$$

meaning a local update that helps one task hurts another. PCGrad removes the conflicting component:

$$
g_i
\leftarrow
g_i-\frac{g_i^\top g_j}{\lVert g_j\rVert_2^2}g_j
\quad\text{when }g_i^\top g_j<0.
$$

<div class="diagram-scroll">

![Gradient conflict and multi-task responses.](assets/multitask-gradient-interference.svg){fig-alt="Two task gradients point in conflicting directions, with loss weighting, GradNorm, PCGrad projection, and task-specific routing shown as mitigation strategies."}

</div>

<details>
<summary><strong>Python: project conflicting gradients with PCGrad</strong></summary>

```python
import numpy as np

gradient_a = np.array([1.0, 0.2, -0.3])
gradient_b = np.array([-0.8, 0.4, 0.1])

def pcgrad_project(gradient, other):
    dot = float(gradient @ other)
    if dot < 0:
        gradient = gradient - dot / (other @ other) * other
    return gradient

projected_a = pcgrad_project(gradient_a.copy(), gradient_b)
plain_update = gradient_a + gradient_b
pcgrad_update = projected_a + gradient_b

print("original dot product:", round(float(gradient_a @ gradient_b), 3))
print("dot after projection:", round(float(projected_a @ gradient_b), 8))
print("plain joint gradient:", np.round(plain_update, 3).tolist())
print("PCGrad joint gradient:", np.round(pcgrad_update, 3).tolist())
```

</details>

Projection is local and order-dependent when many tasks are processed. Removing all conflict can also discard productive trade-offs. Multi-task learning is a multi-objective problem: report every task, Pareto trade-offs, gradient statistics, and single-task baselines rather than only the average score.


### **Continual Learning**

Continual learning updates a model as data, domains, or tasks arrive sequentially. It differs from ordinary online learning by explicitly evaluating retention and transfer across experiences. Common scenarios are:

- **task-incremental:** task identity is available and task-specific heads may be used;
- **domain-incremental:** the task remains the same but the input distribution changes;
- **class-incremental:** new classes arrive and task identity is unavailable at inference;
- **streaming continual learning:** boundaries may be unknown and memory is constrained.

#### **Catastrophic Forgetting**

**Catastrophic forgetting** occurs when optimizing a new task overwrites parameters or representations needed by earlier tasks. Let $A_{i,j}$ be performance on task $j$ after training through task $i$. After $T$ tasks,

$$
\text{Average accuracy}
=\frac{1}{T}\sum_{j=1}^{T}A_{T,j},
$$

and one forgetting measure is

$$
F
=\frac{1}{T-1}\sum_{j=1}^{T-1}
\left(
\max_{i<T}A_{i,j}-A_{T,j}
\right).
$$

Forward transfer asks whether earlier tasks improve learning a later one; backward transfer asks whether later learning improves or harms earlier tasks. Retention must be measured throughout the sequence, not just at the end.

<div class="diagram-scroll">

![Sequential tasks, catastrophic forgetting, and defense strategies.](assets/continual-learning-strategies.svg){fig-alt="Tasks arrive sequentially and later updates can erase earlier solutions; replay, parameter regularization, dynamic architectures, and an accuracy matrix are shown as responses."}

</div>

#### **Replay, Regularization, and Dynamic Architectures**

Three broad families trade memory, privacy, capacity, and interference:

- **Replay:** mix stored old examples, compressed exemplars, or generated samples with current data. Replay is direct and often strong, but storage may violate privacy or licensing constraints.
- **Regularization:** protect important parameters or behavior. Elastic weight consolidation (EWC), for example, penalizes movement from previous parameters $\theta^\star$:

$$
\mathcal L_{\text{new}}(\theta)
+\frac{\lambda}{2}\sum_k F_k(\theta_k-\theta_k^\star)^2,
$$

where $F_k$ approximates parameter importance through the diagonal Fisher information. Distillation instead preserves old outputs or features.
- **Dynamic architectures:** allocate new modules, masks, adapters, or experts and route examples by task or context. They reduce overwrite but increase capacity and may require task identification.

```text
for each new experience t:
    receive current data D_t
    sample replay memory M
    update model on D_t union M
    evaluate every previously seen task
    update M under memory, privacy, and diversity constraints
```

<details>
<summary><strong>Python: measure forgetting with and without replay</strong></summary>

```python
import copy
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn

torch.manual_seed(167)
rng = np.random.default_rng(167)
X, y = load_digits(return_X_y=True)
X = (X / 16.0).astype("float32")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=167
)
permutation = rng.permutation(X.shape[1])

task_a_train = torch.tensor(X_train)
task_a_test = torch.tensor(X_test)
task_b_train = torch.tensor(X_train[:, permutation].copy())
task_b_test = torch.tensor(X_test[:, permutation].copy())
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

def new_model():
    return nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

def train(model, X_data, y_data, epochs=55, lr=0.03):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(X_data), y_data)
        loss.backward()
        optimizer.step()

def accuracy(model, X_data, y_data):
    with torch.no_grad():
        return float((model(X_data).argmax(1) == y_data).float().mean())

initial = new_model().state_dict()

# Sequential training without replay.
plain = new_model()
plain.load_state_dict(copy.deepcopy(initial))
train(plain, task_a_train, y_train_tensor)
a_before = accuracy(plain, task_a_test, y_test_tensor)
train(plain, task_b_train, y_train_tensor)

# Repeat from the same initialization and replay 300 task-A examples.
replay = new_model()
replay.load_state_dict(copy.deepcopy(initial))
train(replay, task_a_train, y_train_tensor)
memory_id = rng.choice(len(task_a_train), 300, replace=False)
mixed_X = torch.cat([task_b_train, task_a_train[memory_id]])
mixed_y = torch.cat([y_train_tensor, y_train_tensor[memory_id]])
train(replay, mixed_X, mixed_y)

print("task A before task B:", round(a_before, 3))
print("no replay - task A / task B:", round(accuracy(plain, task_a_test, y_test_tensor), 3), round(accuracy(plain, task_b_test, y_test_tensor), 3))
print("with replay - task A / task B:", round(accuracy(replay, task_a_test, y_test_tensor), 3), round(accuracy(replay, task_b_test, y_test_tensor), 3))
```

</details>

The second task permutes pixels while preserving digit labels. One network has enough capacity for both mappings, but sequential optimization can overwrite the first. Replay changes the training distribution so both mappings remain represented. A complete benchmark should repeat task orders and seeds, report memory and compute, and prevent test data from entering replay.


### **Meta-Learning and Few-Shot Learning**

Few-shot learning asks a model to solve a new task from a small **support set**. Meta-learning trains across a distribution of previous tasks so the learner acquires an embedding, initialization, optimizer, or inference procedure that adapts efficiently.

In an $N$-way $K$-shot classification episode, the support set contains $K$ labeled examples for each of $N$ classes. A separate query set measures within-episode generalization. Meta-training and meta-testing should use disjoint classes, domains, users, or tasks according to the scientific claim.

<div class="diagram-scroll">

![Episodic meta-training and held-out few-shot evaluation.](assets/episodic-meta-learning.svg){fig-alt="Many meta-training episodes teach a reusable metric or initialization; a held-out meta-test task uses a small support set for adaptation and a separate query set for evaluation."}

</div>

#### **Metric-Based Methods**

Metric-based methods learn an embedding where a small support set defines a classifier directly. In a prototypical network, class $k$ has prototype

$$
c_k
=\frac{1}{|S_k|}
\sum_{(x_i,y_i)\in S_k}f_\theta(x_i),
$$

and a query probability is

$$
P(y=k\mid x)
=
\frac{\exp(-d(f_\theta(x),c_k))}
{\sum_j\exp(-d(f_\theta(x),c_j))}.
$$

The embedding is trained episodically so query loss after prototype construction improves. Euclidean distance corresponds to spherical class clusters in the learned space; one prototype can be insufficient for multimodal classes.

<details>
<summary><strong>Python: evaluate an N-way K-shot prototype classifier</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(168)
X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
classes = np.array([5, 6, 7, 8, 9])  # treated as held-out meta-test classes

def prototype_episode(shots=5, queries_per_class=20):
    support_X, support_y, query_X, query_y = [], [], [], []
    for class_id in classes:
        class_index = np.flatnonzero(y == class_id)
        chosen = rng.choice(class_index, shots + queries_per_class, replace=False)
        support_X.append(X[chosen[:shots]])
        support_y.extend([class_id] * shots)
        query_X.append(X[chosen[shots:]])
        query_y.extend([class_id] * queries_per_class)

    support_X = np.vstack(support_X)
    query_X = np.vstack(query_X)
    query_y = np.array(query_y)
    prototypes = np.vstack([
        support_X[np.array(support_y) == class_id].mean(axis=0)
        for class_id in classes
    ])
    squared_distance = ((query_X[:, None, :] - prototypes[None, :, :]) ** 2).sum(axis=2)
    prediction = classes[squared_distance.argmin(axis=1)]
    return np.mean(prediction == query_y)

for shots in [1, 3, 5, 10]:
    episode_accuracy = np.array([prototype_episode(shots=shots) for _ in range(200)])
    print(
        f"{shots}-shot mean accuracy:",
        round(float(episode_accuracy.mean()), 3),
        "95% interval:",
        np.round(np.quantile(episode_accuracy, [0.025, 0.975]), 3).tolist(),
    )
```

</details>

This uses standardized pixels rather than a meta-learned encoder, so it is a transparent prototype baseline. A proper prototypical network would learn $f_\theta$ on disjoint meta-training classes and preserve the support/query separation shown here.

#### **Optimization-Based Meta-Learning**

Optimization-based methods learn parameters that become useful after a small number of target updates. Model-agnostic meta-learning (MAML) performs an inner update for each task $\tau$:

$$
\theta_\tau'
=\theta-\alpha\nabla_\theta
\mathcal L_{\tau}^{\text{support}}(\theta),
$$

then optimizes the shared initialization using query loss:

$$
\min_\theta
\sum_{\tau\sim p(\tau)}
\mathcal L_{\tau}^{\text{query}}(\theta_\tau').
$$

Differentiating through the inner update introduces second-order derivatives; first-order variants omit them. MAML learns an initialization that is easy to adapt, not one model that is already optimal for every task.

<details>
<summary><strong>Python: implement a compact MAML loop for sine-wave regression</strong></summary>

```python
import numpy as np
import torch

torch.manual_seed(169)
rng = np.random.default_rng(169)

parameters = {
    "w1": torch.randn(1, 32, requires_grad=True) * 0.15,
    "b1": torch.zeros(32, requires_grad=True),
    "w2": torch.randn(32, 32, requires_grad=True) * 0.15,
    "b2": torch.zeros(32, requires_grad=True),
    "w3": torch.randn(32, 1, requires_grad=True) * 0.15,
    "b3": torch.zeros(1, requires_grad=True),
}
# Multiplication above creates non-leaf tensors, so replace them with leaf Parameters.
parameters = {name: torch.nn.Parameter(value.detach()) for name, value in parameters.items()}
optimizer = torch.optim.Adam(parameters.values(), lr=0.004)

def forward(x, p):
    hidden = torch.tanh(x @ p["w1"] + p["b1"])
    hidden = torch.tanh(hidden @ p["w2"] + p["b2"])
    return hidden @ p["w3"] + p["b3"]

def sample_task(support_points, query_points):
    amplitude = rng.uniform(0.2, 5.0)
    phase = rng.uniform(0, np.pi)
    support_x = rng.uniform(-5, 5, size=(support_points, 1)).astype("float32")
    query_x = rng.uniform(-5, 5, size=(query_points, 1)).astype("float32")
    support_y = (amplitude * np.sin(support_x + phase)).astype("float32")
    query_y = (amplitude * np.sin(query_x + phase)).astype("float32")
    return (
        torch.tensor(support_x),
        torch.tensor(support_y),
        torch.tensor(query_x),
        torch.tensor(query_y),
    )

inner_rate = 0.01
for _ in range(320):
    meta_loss = 0.0
    for _ in range(6):
        support_x, support_y, query_x, query_y = sample_task(10, 20)

        support_loss = torch.mean((forward(support_x, parameters) - support_y) ** 2)
        gradient = torch.autograd.grad(
            support_loss, tuple(parameters.values()), create_graph=True
        )
        adapted = {
            name: value - inner_rate * grad
            for (name, value), grad in zip(parameters.items(), gradient)
        }
        meta_loss = meta_loss + torch.mean((forward(query_x, adapted) - query_y) ** 2)

    optimizer.zero_grad()
    (meta_loss / 6).backward()
    optimizer.step()

# Evaluate one unseen task before and after five support-set updates.
amplitude, phase = 3.2, 0.7
support_x = torch.linspace(-4, 4, 10).reshape(-1, 1)
support_y = amplitude * torch.sin(support_x + phase)
query_x = torch.linspace(-5, 5, 200).reshape(-1, 1)
query_y = amplitude * torch.sin(query_x + phase)
adapted = dict(parameters)
with torch.no_grad():
    before = torch.mean((forward(query_x, adapted) - query_y) ** 2).item()
for _ in range(5):
    loss = torch.mean((forward(support_x, adapted) - support_y) ** 2)
    gradient = torch.autograd.grad(loss, tuple(adapted.values()))
    adapted = {
        name: value - inner_rate * grad
        for (name, value), grad in zip(adapted.items(), gradient)
    }
with torch.no_grad():
    after = torch.mean((forward(query_x, adapted) - query_y) ** 2).item()

print("query MSE before adaptation:", round(before, 3))
print("query MSE after five updates:", round(after, 3))
```

</details>

Few-shot evaluation is statistically demanding. Report many held-out episodes, confidence intervals, the support-selection process, whether class identities are novel, and strong non-meta baselines such as pretrained embeddings plus logistic regression or nearest prototypes.


### **Evaluating Adaptation and Transfer**

Transfer methods are easy to evaluate optimistically because source and target information can leak through preprocessing, hyperparameter selection, pretrained checkpoints, or domain construction. The split must match the claim:

- hold out users to claim new-user transfer;
- hold out hospitals or devices to claim new-site transfer;
- use chronological splits to claim future robustness;
- hold out classes or tasks to claim few-shot meta-generalization;
- repeat which domain is held out to measure source sensitivity.

#### **Held-Out Domains and Transfer Baselines**

<div class="diagram-scroll">

![Held-out-domain evaluation and required transfer baselines.](assets/transfer-evaluation-matrix.svg){fig-alt="Several source domains train and select a model before one domain is held out for evaluation against source-only, target-only, pooled, frozen-probe, and adaptation baselines."}

</div>

At minimum, compare:

1. **source only:** deploy the source model unchanged;
2. **target only / scratch:** use exactly the available target labels without source knowledge;
3. **pooled data:** combine source and permitted target labels without specialized adaptation;
4. **frozen probe:** transfer representations but not encoder updates;
5. **fine-tuning or adaptation:** the proposed method;
6. **target oracle:** abundant target labels, clearly marked as an upper reference rather than a fair competitor.

Target-only performance is the reference for positive transfer. Source-only performance measures the unadapted gap. Pooled training establishes whether a complex method beats simple data combination.

<details>
<summary><strong>Python: compare random splits with held-out-domain validation</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(170)
features, labels, domains = [], [], []
for domain_id in range(4):
    n = 900
    core = rng.normal(size=n)
    y = (core + rng.normal(scale=0.9, size=n) > 0).astype(int)
    # A spurious feature reverses only in the held-out fourth domain.
    direction = 1.0 if domain_id < 3 else -1.0
    spurious = direction * (2 * y - 1) + rng.normal(scale=0.45, size=n)
    nuisance = rng.normal(loc=0.6 * domain_id, scale=1.0, size=n)
    X_domain = np.column_stack([core, spurious, nuisance])
    features.append(X_domain)
    labels.append(y)
    domains.append(np.full(n, domain_id))

X = np.vstack(features)
y = np.concatenate(labels)
domain = np.concatenate(domains)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

random_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=170)
random_score = cross_val_score(model, X, y, cv=random_cv, scoring="accuracy")

held_out_score = []
for target_domain in np.unique(domain):
    source = domain != target_domain
    target = domain == target_domain
    fitted = make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)
    ).fit(X[source], y[source])
    held_out_score.append(accuracy_score(y[target], fitted.predict(X[target])))

print("random-row CV accuracy:", round(float(random_score.mean()), 3))
print("held-out-domain accuracy:", np.round(held_out_score, 3).tolist())
print("held-out-domain mean:", round(float(np.mean(held_out_score)), 3))
```

</details>

Random splitting lets every domain influence training and hides the failure on the domain where the spurious relationship reverses. The domain-level result also reveals heterogeneity that one average conceals.

Performance tables should include target score, transfer gain, target-label budget, source domains, target information used during training, model-selection protocol, calibration, subgroup metrics, compute, and memory. For continual learning, report the full accuracy matrix $A_{i,j}$, not only final average accuracy. For few-shot learning, use episode-level intervals and class-disjoint meta-test tasks.

<details>
<summary><strong>Python: compute transfer gain and continual-learning summaries</strong></summary>

```python
import numpy as np

# Rows: after learning tasks 1, 2, and 3. Columns: evaluation tasks 1, 2, and 3.
accuracy_matrix = np.array([
    [0.91, np.nan, np.nan],
    [0.84, 0.88, np.nan],
    [0.79, 0.85, 0.90],
])
final_average_accuracy = np.nanmean(accuracy_matrix[-1])
forgetting = np.mean([
    np.nanmax(accuracy_matrix[:, task]) - accuracy_matrix[-1, task]
    for task in range(accuracy_matrix.shape[1] - 1)
])

target_only_score = np.array([0.62, 0.71, 0.78])
transfer_score = np.array([0.74, 0.77, 0.80])
transfer_gain = transfer_score - target_only_score

print("final average accuracy:", round(float(final_average_accuracy), 3))
print("mean forgetting:", round(float(forgetting), 3))
print("transfer gain by target-label budget:", np.round(transfer_gain, 3).tolist())
```

</details>

Uncertainty should respect the unit of generalization. Bootstrap domains when the claim concerns new domains, tasks when it concerns new tasks, users for new users, and episodes for few-shot evaluation. Bootstrapping individual rows cannot recover uncertainty across only a few hospitals or environments.

### **Choosing What to Share**

Sharing is a statistical and operational decision. The most reusable component is the one supported by an invariance that remains plausible in the target:

| Evidence and constraint | Sensible starting point | Main failure to test |
|---|---|---|
| Same task, covariate shift, good support overlap | Instance weighting | Extreme weights and unsupported target regions |
| Same semantics, transformed feature statistics | CORAL, MMD, or conditional alignment | Aligning different classes together |
| Large related source task and few target labels | Frozen probe, then gradual fine-tuning | Source bias and overfitting |
| Several simultaneous related outputs | Shared trunk with task-specific heads | Gradient conflict and task domination |
| Sequential tasks with replay permitted | Small representative replay buffer | Privacy, memory, and biased memory selection |
| Sequential tasks without data storage | EWC, distillation, adapters, or routing | Weak protection or capacity growth |
| Repeated family of few-shot tasks | Metric or optimization-based meta-learning | Meta-test leakage and weak simple baselines |
| Uncertain relatedness | Target-only baseline plus selective or gated transfer | Negative transfer hidden by pooled averages |

A defensible workflow is:

1. define source and target domain-task pairs;
2. identify the assumed invariant and target information allowed;
3. build target-only, source-only, and pooled baselines;
4. begin with the weakest form of sharing that could help;
5. measure transfer gain over target-label budgets and held-out domains;
6. inspect weights, feature alignment, gradient conflict, forgetting, and subgroup failures;
7. increase sharing only when target validation consistently supports it;
8. retain a rollback path when deployment evidence contradicts the assumption.

Transfer is successful when it improves the target problem under a fair resource comparison, not when a pretrained checkpoint is involved. The safest default is selective sharing with explicit diagnostics and a target-only escape route.

Primary and official resources include [Dataset Shift in Machine Learning](https://doi.org/10.7551/mitpress/9780262170055.001.0001), the [label-shift correction paper](https://proceedings.mlr.press/v80/lipton18a.html), the [domain-adversarial training paper](https://jmlr.org/papers/v17/15-239.html), [impossibility results for domain adaptation](https://proceedings.mlr.press/v9/david10a.html), the original [multi-task learning paper](https://doi.org/10.1023/A:1007379606734), [GradNorm](https://proceedings.mlr.press/v80/chen18a.html), [PCGrad](https://papers.nips.cc/paper/2020/hash/3fe78a8acf5fda99de95303940a2420c-Abstract.html), [elastic weight consolidation](https://doi.org/10.1073/PNAS.1611835114), [prototypical networks](https://papers.nips.cc/paper/2017/hash/cb8da6767461f2812ae4290eac7cbc42-Abstract.html), and [MAML](https://proceedings.mlr.press/v70/finn17a.html).
